# TF-IDF Text Baseline (IEMOCAP)

Train a text-only baseline from IEMOCAP transcriptions using a session split
(Sessions 1-4 train, Session 5 test).


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Resolve dataset paths relative to repository root.
repo_root = Path.cwd().parents[1]
meta_csv = repo_root / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
iemocap_root = repo_root / "datasets" / "IEMOCAP"
meta_csv


WindowsPath('f:/Speech-Emotion-Recognition/datasets/IEMOCAP/iemocap_full_dataset.csv')

In [2]:
df = pd.read_csv(meta_csv)
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()
# Keep rows that are labeled (not xxx) and have at least some annotator agreement.
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape


(7532, 7)

In [3]:
# Parse transcript lines like:
# Ses01F_script02_1_F000 [015.1400-017.2100]: Fine.
# Groups:
# - utt: utterance ID token before the timestamp
# - text: spoken text after ":"
line_re = re.compile(r"^(?P<utt>\S+)\s+\[[^\]]+\]:\s*(?P<text>.*)$")

def build_transcript_index(iemocap_dir: Path) -> dict[str, str]:
    # Build utterance_id -> transcript text by scanning all transcription files.
    idx: dict[str, str] = {}
    for txt in iemocap_dir.glob("Session*/dialog/transcriptions/*.txt"):
        with txt.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                m = line_re.match(line.strip())
                if not m:
                    continue
                utt = m.group("utt")
                text = m.group("text").strip()
                if not text:
                    continue
                # Some utterances may appear multiple times; concatenate to preserve content.
                if utt in idx:
                    idx[utt] = (idx[utt] + " " + text).strip()
                else:
                    idx[utt] = text
    return idx

transcripts = build_transcript_index(iemocap_root)
len(transcripts)


10084

In [4]:
# Derive utterance ID from wav path stem and join with transcript text.
df["utt_id"] = df["path"].apply(lambda p: Path(p).stem)
df["text"] = df["utt_id"].map(transcripts)

missing_text = df["text"].isna().sum()
print(f"Rows with missing transcript text: {missing_text}")

# Keep only rows with non-empty text for text-only modeling.
df_text = df[df["text"].notna() & (df["text"].str.len() > 0)].copy()
df_text.shape


Rows with missing transcript text: 0


(7532, 9)

In [5]:
# Session-based split to match existing audio experiments.
train_mask = df_text["session"].isin([1, 2, 3, 4])
test_mask = df_text["session"] == 5

X_train_text = df_text.loc[train_mask, "text"]
y_train = df_text.loc[train_mask, "emotion"]
X_test_text = df_text.loc[test_mask, "text"]
y_test = df_text.loc[test_mask, "emotion"]

X_train_text.shape, X_test_text.shape


((5882,), (1650,))

In [6]:
# TF-IDF baseline: unigrams+bigrams, drop very rare terms.
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
)

# Linear classifier baseline for sparse TF-IDF features.
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42,
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1:  {f1:.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.5067
Macro F1:  0.3455

Classification report:

              precision    recall  f1-score   support

         ang       0.51      0.67      0.58       170
         dis       0.00      0.00      0.00         0
         exc       0.57      0.41      0.48       299
         fea       0.15      0.30      0.20        10
         fru       0.57      0.59      0.58       381
         hap       0.41      0.31      0.35       143
         neu       0.53      0.47      0.49       384
         oth       0.00      0.00      0.00         0
         sad       0.57      0.56      0.56       245
         sur       0.13      0.72      0.22        18

    accuracy                           0.51      1650
   macro avg       0.34      0.40      0.35      1650
weighted avg       0.53      0.51      0.51      1650



f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [7]:
# Cross-validation on the train split only.
pipe = make_pipeline(
    TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=20000),
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=42),
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(pipe, X_train_text, y_train, cv=skf, scoring="accuracy", n_jobs=-1)
cv_f1 = cross_val_score(pipe, X_train_text, y_train, cv=skf, scoring="f1_macro", n_jobs=-1)

print(f"CV Accuracy (mean+-std): {np.mean(cv_acc):.4f} +- {np.std(cv_acc):.4f}")
print(f"CV Macro F1 (mean+-std):  {np.mean(cv_f1):.4f} +- {np.std(cv_f1):.4f}")


f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


CV Accuracy (mean+-std): 0.5099 +- 0.0093
CV Macro F1 (mean+-std):  0.3960 +- 0.0206
